# ReAct = Reasoning + Acting

Reasoning: Agent decide karta hai ki question solve karne ke liye kya karna chahiye.
Acting: Agent koi action leta hai, jaise calculator, search tool ya database tool call karna.


Yaani ReAct agent zarurat padne par external tools use kar sakta hai.

## ReAct Agent Architecture

![ReAct Agent Architecture](../images/react-agent.png)


ReAct Agent : 	Decide karta hai kya action lena hai
Tool Binding : 	Model ko available tools ki information dena
Toolkit : 	Actual tools ka collection


Case 1: LLM khud answer de sakta hai

User: "What is Python?"
- LLM question understand karega.
- External tool ki zarurat nahi hai.
- Direct response generate karega.
Case 2: External tool ki zarurat hai

User: "Mere order 105 ka status check karo."
- Reasoning: Agent decide karega ki database tool use karna hai.
- Acting: Database tool execute hoga.
- Observation: Database se result wapas aayega.

# Tool execute hone ke baad kya hota hai?

Maan le user ne bola:

"Mera order check karo aur agar shipped hai toh confirmation email bhejo."

Agent ke paas do tools hain:
- get_order_status()
- send_email()




![Tool Execution Architecture](../images/tool-execution.png)

Database tool execute hua, lekin pura task complete nahi hua. Toh kya ReAct Agent fir se check karega?

YES! Exactly!

Tool ka result wapas LLM/agent ko diya jata hai. Agent us observation ko dekhta hai aur decide karta hai:
- Task complete ho gaya? → Final answer.
- Ek aur tool chahiye? → Next action.
- Aur information chahiye? → Another tool call.
- Tool fail ho gaya? → Situation ke according retry ya error response.
Yahi ReAct ka iterative loop hai:


![retry Architecture](../images/image.png)

ReAct Agent ek baar tool call karke rukta nahi hai. Tool ka result observe karke dobara decide karta hai ki task complete hua ya aur action lena hai.

# Tools



In [ ]:
# Tool -1 [News Search Tool]

# install : uv add -U langchain-community duckduckgo-search

from langchain_community.tools import DuckDuckGoSearchRun

# Simple string output
search_tool = DuckDuckGoSearchRun()
print(search_tool.invoke("latest LangChain release"))

#jo answer ayega woh response of search on ddg , it is not genrated by the LLm (mind these abhi llm scene mein hi nhai hain)



In [ ]:
#Tool-2 [Wikipedia Search TOol]

#install : uv add langchain-community wikipedia

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# 1. Configure the API wrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000)

# 2. Initialize the tool
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper,description = "This is a tool to search Wikipedia")

# 3. Run a query directly
result = wiki_tool.invoke({"query": "LangChain"})
print(result)




In [ ]:
#Tool -3 {Custom Tool}
#Custom Enterprise Tool

from langchain.tools import tool

@tool
def enterprise_tool(query:str)->str:
    """This is a tool to send Emails to Employees"""
    return "Email Sent"


In [ ]:
ToolKit =[search_tool,wiki_tool,enterprise_tool]

In [ ]:
ToolKit

# ReAct Agent


In [ ]:
from langchain_openai import ChatOpenAI

#Step 1. Pehle model initialize karo:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=500,
    timeout=30
)

Yahan:
- model: Kaunsa LLM use karna hai.
- temperature: Response kitna creative ho.
- max_tokens: Response ki maximum length.
- timeout: Request ke liye timeout limit in seconds.

#Step 2. Agent create karo:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=ToolKit,
    system_prompt="You are a helpful assistant. Use tools whenever required."
)

# #Step 3. Agent ko input do:
# response = agent.invoke({
#     "messages": [
#         {"role": "user", "content": "Calculate 25 multiplied by 48"}
#     ]
# })


#step 3 

#Streaming Data : shows data as soon as it arrives
# {we see what is going on background}

# Invoke ki jagah stream use kar rahe hain

query = "Give me the latest news about the stock market"

events = agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()


#Streaming ke modes samajh
Mode	Kya milta hai?
updates: 	Har agent/node execution ka update
messages: 	LLM ke message chunks aur metadata
values: 	Har step ke baad complete graph state

# Sql React Agent

User poochta hai:

"Show me employees earning more than 50,000."


![Sql ReAct Agent Architecture](../images/sql-react-agent.png)

In [ ]:
# setup_db

import sqlite3

# Create database
conn = sqlite3.connect("company.db")

cursor = conn.cursor()

# Create employees table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")

# Insert sample data
cursor.executemany("""
INSERT INTO employees (id, name, department, salary)
VALUES (?, ?, ?, ?)
""", [
    (1, "Rahul", "IT", 60000),
    (2, "Priya", "HR", 45000),
    (3, "Aman", "IT", 80000),
    (4, "Neha", "Finance", 55000)
])

conn.commit()
conn.close()

print("Database created successfully!")

In [ ]:
python setup_db.py

In [ ]:
# 4. SQL Toolkit create karo
#LangChain ka SQLDatabaseToolkit ready-made database tools provide karta hai.


from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# 1. Connect to database
db = SQLDatabase.from_uri("sqlite:///company.db")

# 2. Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. Create SQL Toolkit
toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

# 4. Get available tools
tools = toolkit.get_tools()

# 5. Print available tools
for tool in tools:
    print(tool.name, ":", tool.description)


5. Toolkit mein kaunse tools milenge?
Typically ye tools milte hain:
Tool	Kaam
sql_db_list_tables:	Database mein tables ki list
sql_db_schema:	Table ka schema aur sample rows
sql_db_query:	SQL query execute karna
sql_db_query_checker:	Query execute karne se pehle check karna

# Ab samajh: Agent ko manually SQL query likhkar dene ki zarurat nahi. Toolkit ke tools available hain, agent unhe use kar sakta hai.



In [ ]:
# 6. ReAct SQL Agent create karo

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a SQL database assistant.

    Use the available SQL tools to answer user questions.

    First understand the database schema.
    Then generate and execute the appropriate SQL query.
    Use the query checker when appropriate.

    Only perform read-only queries.
    Never modify or delete database records.

    Explain the result in simple language.
    """
)

In [ ]:
# 7. Agent ko question do — Streaming

query = "Show me all employees earning more than 50000"

events = agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values"
)

for event in events:
    event["messages"][-1].pretty_print()


#expected result : Expected result: Rahul, Aman aur Neha ki details. 


SQL query LLM hi generate karta hai. 🔥

SQL Toolkit query generate nahi karta; woh LLM ko database ke saath interact karne ke tools provide karta hai.

Chal example dekh.

User asks:
Which department has the highest average salary?

Step-by-step kya hota hai?
1. User → Agent

Agent ko natural language question milta hai.

2. LLM → Schema check karne ka decision

LLM decide karta hai ki pehle database ka structure dekhna chahiye.

Tool call:
sql_db_schema("employees")

3. Database → Schema return karta hai
employees(
    id,
    name,
    department,
    salary
)

4. LLM → SQL query generate karta hai

Ab LLM schema samajh kar SQL query banata hai:

SELECT department, AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC
LIMIT 1;

5. SQL Tool → Query execute karta hai

Database query run karta hai aur result return karta hai:

IT | 70000


6. LLM → Final answer

"The IT department has the highest average salary, with an average salary of 70,000."





Component  	Responsibility
LLM :	SQL query generate karna
SQL Toolkit : 	Database-related tools provide karna
SQL Tool : 	Generated query execute karna
Database : 	Actual data aur query result provide karna
ReAct Agent	Decide karna ki next kaunsa tool use karna hai

Aur haan: Agar query galat generate hui, toh agent query checker use kar sakta hai, error observe karke query correct karne ki koshish bhi kar sakta hai.